<a href="https://colab.research.google.com/github/PatrickMatheus/Impact_Lab/blob/main/Atividades_PPH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Primeiro código em CUDA

In [ ]:
code = r"""
#include <iostream>
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloFromGPU(){
    printf("Hello from CUDA GPU!\n");
}

int main(){

  helloFromGPU<<<1,1>>>();
  cudaDeviceSynchronize();

  printf("Hello from CPU!\n");

  cudaDeviceReset();

  return 0;
}


"""


Compilando

In [ ]:
with open("hello.cu", "w") as f:
    f.write(code)

# Compila com nvcc
!nvcc -arch=sm_75 hello.cu -o hello


# Executa o binário
!./hello

Usando threads para somar vetores

In [ ]:
code2 = r"""
#include <iostream>
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void add (int* a, int* b, int* c){
  int idx = threadIdx.x;
  c[idx] = a[idx] + b[idx];
  printf("Hello from CUDA thread X %d\n", threadIdx.x);
}

int main(int argc, char** argv){
  int N = 10;
  int size = N * sizeof(int);

  int h_a[N], h_b[N], h_c[N];

  int* d_a, *d_b, *d_c;

  cudaMalloc((void**)&d_a, size);
  cudaMalloc((void**)&d_b, size);
  cudaMalloc((void**)&d_c, size);

  for (int i = 0; i < N; ++i) {
		  h_a[i] = i + 1;
		  h_b[i] = i + 2;
	  }

  cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

  std::cout << "Hello Wold From CPU !\n";
  add << <1,N >> > (d_a, d_b, d_c);
  cudaDeviceSynchronize();

  printf("Hello from GPU (CPU Main)!\n");

  cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

  for (int i = 0; i < N; ++i) {
		std::cout << h_a[i] << " + " << h_b[i] << " = " << h_c[i] << std::endl;
	}

  cudaFree(d_a);
	cudaFree(d_b);
	cudaFree(d_c);

  return 0;
}
"""

Compilando

In [ ]:
with open("add.cu", "w") as f:
    f.write(code2)

# Compila com nvcc
!nvcc -arch=sm_75 add.cu -o add


# Executa o binário
!./add

Somando vetores usando blocos

In [ ]:
code3 = r"""
#include <iostream>
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void add_block(int* a, int* b, int* c){
  c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
  printf("Hello from CUDA Block X %d\n", blockIdx.x);
	printf("Hello from CUDA thread X %d\n", threadIdx.x);
}

int main(int argc, char** argv){
  int N = 10;
  int size = N * sizeof(int);

  int h_a[N], h_b[N], h_c[N];

  int* d_a, *d_b, *d_c;

  cudaMalloc((void**)&d_a, size);
  cudaMalloc((void**)&d_b, size);
  cudaMalloc((void**)&d_c, size);

  for (int i = 0; i < N; ++i) {
		  h_a[i] = i + 1;
		  h_b[i] = i + 2;
	  }

  cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

  std::cout << "Hello Wold From CPU !\n";

  add_block << <N,1 >> > (d_a, d_b, d_c);
  cudaDeviceSynchronize();

  printf("Hello from GPU (CPU Main)!\n");

  cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

  for (int i = 0; i < N; ++i) {
		std::cout << h_a[i] << " + " << h_b[i] << " = " << h_c[i] << std::endl;
	}

  cudaFree(d_a);
	cudaFree(d_b);
	cudaFree(d_c);

  return 0;
}


"""

Compilando

In [ ]:
with open("add_block.cu", "w") as f:
    f.write(code3)

# Compila com nvcc
!nvcc -arch=sm_75 add_block.cu -o add_block

# Executa o binário
!./add_block

Somando vetores usando blocos e threads

In [ ]:
code4 = r"""
#include <iostream>
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void add_block_thread (int* a, int* b, int* c){
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  c[idx] = a[idx] + b[idx];
  printf("Hello from CUDA block X %d thread X %d\n", blockIdx.x, threadIdx.x);
}

int main(int argc, char** argv){
  int N = 2000;
  int size = N * sizeof(int);

  int h_a[N], h_b[N], h_c[N];

  int* d_a, *d_b, *d_c;

  cudaMalloc((void**)&d_a, size);
  cudaMalloc((void**)&d_b, size);
  cudaMalloc((void**)&d_c, size);

  for (int i = 0; i < N; ++i) {
		  h_a[i] = i + 1;
		  h_b[i] = i + 2;
	  }

  cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

  std::cout << "Hello Wold From CPU !\n";

  int threadsPerBlock = 256;
  int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

  add_block_thread << <blocksPerGrid,threadsPerBlock >> > (d_a, d_b, d_c);
  cudaDeviceSynchronize();

  printf("Hello from GPU (CPU Main)!\n");

  cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

  for (int i = 0; i < N; ++i) {
		std::cout << h_a[i] << " + " << h_b[i] << " = " << h_c[i] << std::endl;
	}

  cudaFree(d_a);
	cudaFree(d_b);
	cudaFree(d_c);

  return 0;
}
"""

Compilando

In [ ]:
with open("add_block_thread.cu", "w") as f:
    f.write(code4)

# Compila com nvcc
!nvcc -arch=sm_75 add_block_thread.cu -o add_block_thread

# Executa o binário
!./add_block_thread